In [ ]:
# Import libraries
import pandas as pd
import numpy as np
from pathlib import Path
from itertools import combinations
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.model_selection import cross_val_score

print("Libraries imported successfully")

In [ ]:
# Load preprocessed data
data_path = Path.cwd().parent / "shared" / "output" / "production_preprocessed_data.csv"
df = pd.read_csv(data_path, encoding='utf-8-sig')

# Global z-score normalization (across all participants)
df['normed_log_vot'] = (df['log_vot'] - df['log_vot'].mean()) / df['log_vot'].std()
df['normed_semitone'] = (df['semitone'] - df['semitone'].mean()) / df['semitone'].std()

print(f"Loaded {len(df)} rows from preprocessed data")
print(f"Participants: {df['prolific_id'].nunique()}")
print(f"\nColumns: {list(df.columns)}")
print(f"\nPhonation distribution:")
print(df['phonation'].value_counts())
print(f"\nGlobal normalization stats:")
print(f"  log_vot  → mean={df['log_vot'].mean():.4f}, std={df['log_vot'].std():.4f}")
print(f"  semitone → mean={df['semitone'].mean():.4f}, std={df['semitone'].std():.4f}")

In [ ]:
# ================ 참가자별 LDA 결정 경계 시각화 (전체 z-scored, 나이순 정렬) ================
import matplotlib.pyplot as plt
import numpy as np
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score

print("=" * 60)
print("참가자별 LDA 결정 경계 시각화 (전체 z-scored, 나이순 정렬)")
print("=" * 60)

lda_data_individual = df[['semitone', 'log_vot', 'normed_semitone', 'normed_log_vot',
                           'phonation', 'prolific_id', 'age', 'gender']].dropna().copy()

label_alias = {'fortis': 'tense'}
lda_data_individual['phonation_clean'] = (
    lda_data_individual['phonation']
    .astype(str).str.strip().str.lower()
    .replace(label_alias)
)

valid_labels = ['lenis', 'tense', 'aspirated']
unknown_labels = sorted(set(lda_data_individual['phonation_clean']) - set(valid_labels))
if unknown_labels:
    print(f"경고: 예상하지 못한 phonation 라벨이 제외됩니다: {unknown_labels}")
lda_data_individual = lda_data_individual[lda_data_individual['phonation_clean'].isin(valid_labels)].copy()

valid_participants = []
for participant_id in lda_data_individual['prolific_id'].unique():
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id]
    phonation_counts = participant_data['phonation_clean'].value_counts()
    if len(phonation_counts) >= 3 and phonation_counts.min() >= 2:
        valid_participants.append(participant_id)

participant_ages = {
    pid: lda_data_individual[lda_data_individual['prolific_id'] == pid]['age'].iloc[0]
    for pid in valid_participants
}
valid_participants_sorted = sorted(valid_participants, key=lambda x: participant_ages[x])

print(f"\n시각화할 참가자 수: {len(valid_participants_sorted)}명")
print(f"나이 범위: {min(participant_ages.values()):.0f} ~ {max(participant_ages.values()):.0f}세")

colors_dark  = {'lenis': '#1f77b4', 'tense': '#ff7f0e', 'aspirated': '#2ca02c'}
colors_light = {'lenis': '#aec7e8', 'tense': '#ffbb78', 'aspirated': '#98df8a'}
label_to_num = {'lenis': 0, 'tense': 1, 'aspirated': 2}

n_cols = 5
n_rows = int(np.ceil(len(valid_participants_sorted) / n_cols))
print(f"그리드 크기: {n_rows} 행 x {n_cols} 열")

fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 4, n_rows * 4))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for idx, participant_id in enumerate(valid_participants_sorted):
    ax = axes[idx]
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id].copy()

    X = participant_data[['normed_log_vot', 'normed_semitone']].values
    y = participant_data['phonation_clean'].values

    x_range = participant_data['normed_log_vot'].max() - participant_data['normed_log_vot'].min()
    y_range = participant_data['normed_semitone'].max() - participant_data['normed_semitone'].min()
    x_min = participant_data['normed_log_vot'].min() - x_range * 0.05
    x_max = participant_data['normed_log_vot'].max() + x_range * 0.05
    y_min = participant_data['normed_semitone'].min() - y_range * 0.05
    y_max = participant_data['normed_semitone'].max() + y_range * 0.05

    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)

        y_pred   = lda.predict(X)
        accuracy = accuracy_score(y, y_pred)

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        Z = lda.predict(np.c_[xx.ravel(), yy.ravel()])
        Z_numeric = np.array([label_to_num[label] for label in Z]).reshape(xx.shape)

        cmap = plt.matplotlib.colors.ListedColormap([colors_light['lenis'],
                                                     colors_light['tense'],
                                                     colors_light['aspirated']])
        ax.contourf(xx, yy, Z_numeric, alpha=0.3, cmap=cmap, levels=2)
        ax.contour(xx, yy, Z_numeric, colors='black', linewidths=1.5, linestyles='solid', levels=2)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                ax.scatter(participant_data.loc[mask, 'normed_log_vot'],
                           participant_data.loc[mask, 'normed_semitone'],
                           c=colors_dark[phonation], alpha=0.7, s=30,
                           edgecolors='black', linewidth=0.5)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                cx = participant_data.loc[mask, 'normed_log_vot'].mean()
                cy = participant_data.loc[mask, 'normed_semitone'].mean()
                ax.scatter(cx, cy, c=colors_dark[phonation], marker='X', s=150,
                           edgecolors='black', linewidth=2, zorder=5)

        participant_age    = participant_data['age'].iloc[0]
        participant_gender = participant_data['gender'].iloc[0]

        ax.set_xlabel('Norm. log(VOT) (z)', fontsize=9)
        ax.set_ylabel('Norm. Semitone (z)',  fontsize=9)
        ax.set_title(f'P{participant_id} ({participant_age}, {participant_gender})\nAcc: {accuracy*100:.1f}%',
                     fontsize=10, fontweight='bold')
        ax.set_xlim([x_min, x_max])
        ax.set_ylim([y_min, y_max])
        ax.grid(True, alpha=0.3, linewidth=0.5)

    except Exception as e:
        participant_age    = participant_data['age'].iloc[0] if 'age' in participant_data.columns else 'N/A'
        participant_gender = participant_data['gender'].iloc[0] if 'gender' in participant_data.columns else 'N/A'
        ax.text(0.5, 0.5, f'P{participant_id} ({participant_age}, {participant_gender})\nError: {str(e)[:30]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=9)

for idx in range(len(valid_participants_sorted), len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Participant-Level LDA Decision Boundaries — Global Z-scored (Sorted by Age, N={len(valid_participants_sorted)})',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()

output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_filename = output_dir / 'lda_decision_boundaries_by_age_global_zscore.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
pdf_filename = output_dir / 'lda_decision_boundaries_by_age_global_zscore.pdf'
plt.savefig(pdf_filename, bbox_inches='tight', facecolor='white')
print(f"\n✓ 그림이 저장되었습니다: {output_filename}")
print(f"✓ PDF로도 저장되었습니다: {pdf_filename}")
plt.show()

print(f"\n✓ 완료! 총 {len(valid_participants_sorted)}명 | 전체 z-scored (normed_log_vot, normed_semitone) | 나이 오름차순")

In [ ]:
# ================ 참가자별 LDA — 일관된 축 범위 + Equal Aspect ================
# 모든 서브플롯이 동일한 xlim/ylim 공유, set_aspect('equal')로 실제 거리 비율 보존
print("=" * 60)
print("참가자별 LDA 결정 경계 — 일관된 축 범위 + Equal Aspect")
print("=" * 60)

# 전체 데이터 기준 글로벌 범위 (패딩 5%)
gx_min = lda_data_individual['normed_log_vot'].min()
gx_max = lda_data_individual['normed_log_vot'].max()
gy_min = lda_data_individual['normed_semitone'].min()
gy_max = lda_data_individual['normed_semitone'].max()

x_pad = (gx_max - gx_min) * 0.05
y_pad = (gy_max - gy_min) * 0.05
fixed_x_min, fixed_x_max = gx_min - x_pad, gx_max + x_pad
fixed_y_min, fixed_y_max = gy_min - y_pad, gy_max + y_pad

print(f"Global axis limits:")
print(f"  X (normed_log_vot) : [{fixed_x_min:.3f}, {fixed_x_max:.3f}]  (range {fixed_x_max - fixed_x_min:.3f})")
print(f"  Y (normed_semitone): [{fixed_y_min:.3f}, {fixed_y_max:.3f}]  (range {fixed_y_max - fixed_y_min:.3f})")

colors_dark  = {'lenis': '#1f77b4', 'tense': '#ff7f0e', 'aspirated': '#2ca02c'}
colors_light = {'lenis': '#aec7e8', 'tense': '#ffbb78', 'aspirated': '#98df8a'}
label_to_num = {'lenis': 0, 'tense': 1, 'aspirated': 2}

n_cols = 5
n_rows = int(np.ceil(len(valid_participants_sorted) / n_cols))

# equal aspect이므로 서브플롯 가로 크기를 x/y 범위 비율에 맞춤
subplot_h = 4
x_y_ratio = (fixed_x_max - fixed_x_min) / (fixed_y_max - fixed_y_min)
fig, axes = plt.subplots(n_rows, n_cols,
                         figsize=(n_cols * subplot_h * x_y_ratio, n_rows * subplot_h))
if n_rows == 1:
    axes = axes.reshape(1, -1)
axes = axes.flatten()

for idx, participant_id in enumerate(valid_participants_sorted):
    ax = axes[idx]
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id].copy()
    X = participant_data[['normed_log_vot', 'normed_semitone']].values
    y = participant_data['phonation_clean'].values

    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)
        y_pred   = lda.predict(X)
        accuracy = accuracy_score(y, y_pred)

        xx, yy = np.meshgrid(np.linspace(fixed_x_min, fixed_x_max, 200),
                             np.linspace(fixed_y_min, fixed_y_max, 200))
        Z         = lda.predict(np.c_[xx.ravel(), yy.ravel()])
        Z_numeric = np.array([label_to_num[label] for label in Z]).reshape(xx.shape)

        cmap = plt.matplotlib.colors.ListedColormap([colors_light['lenis'],
                                                     colors_light['tense'],
                                                     colors_light['aspirated']])
        ax.contourf(xx, yy, Z_numeric, alpha=0.3, cmap=cmap, levels=2)
        ax.contour(xx, yy, Z_numeric, colors='black', linewidths=1.5, linestyles='solid', levels=2)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                ax.scatter(participant_data.loc[mask, 'normed_log_vot'],
                           participant_data.loc[mask, 'normed_semitone'],
                           c=colors_dark[phonation], alpha=0.7, s=30,
                           edgecolors='black', linewidth=0.5)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                cx = participant_data.loc[mask, 'normed_log_vot'].mean()
                cy = participant_data.loc[mask, 'normed_semitone'].mean()
                ax.scatter(cx, cy, c=colors_dark[phonation], marker='X', s=150,
                           edgecolors='black', linewidth=2, zorder=5)

        participant_age    = participant_data['age'].iloc[0]
        participant_gender = participant_data['gender'].iloc[0]
        ax.set_xlabel('Norm. log(VOT) (z)', fontsize=9)
        ax.set_ylabel('Norm. Semitone (z)',  fontsize=9)
        ax.set_title(f'P{participant_id} ({participant_age}, {participant_gender})\nAcc: {accuracy*100:.1f}%',
                     fontsize=10, fontweight='bold')
        ax.set_xlim([fixed_x_min, fixed_x_max])
        ax.set_ylim([fixed_y_min, fixed_y_max])
        ax.set_aspect('equal', adjustable='box')
        ax.grid(True, alpha=0.3, linewidth=0.5)

    except Exception as e:
        participant_age    = participant_data['age'].iloc[0] if 'age' in participant_data.columns else 'N/A'
        participant_gender = participant_data['gender'].iloc[0] if 'gender' in participant_data.columns else 'N/A'
        ax.text(0.5, 0.5, f'P{participant_id} ({participant_age}, {participant_gender})\nError: {str(e)[:30]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=9)
        ax.set_xlim([fixed_x_min, fixed_x_max])
        ax.set_ylim([fixed_y_min, fixed_y_max])

for idx in range(len(valid_participants_sorted), len(axes)):
    axes[idx].axis('off')

plt.suptitle(f'Participant-Level LDA — Consistent Axes + Equal Aspect (N={len(valid_participants_sorted)})',
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()

output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_filename = output_dir / 'lda_decision_boundaries_by_age_consistent_axes.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
pdf_filename = output_dir / 'lda_decision_boundaries_by_age_consistent_axes.pdf'
plt.savefig(pdf_filename, bbox_inches='tight', facecolor='white')
print(f"\n✓ 저장됨: {output_filename}")
print(f"✓ PDF 저장됨: {pdf_filename}")
plt.show()

# Equidistance analysis

In [ ]:
# ================ Distance Variance Analysis ================
import matplotlib.pyplot as plt
import numpy as np
from scipy.spatial.distance import pdist, squareform

print("=" * 60)
print("Distance Variance Analysis")
print("=" * 60)

lda_data_individual = df[['normed_semitone', 'normed_log_vot',
                           'phonation', 'prolific_id', 'age', 'gender']].dropna().copy()

label_alias = {'fortis': 'tense'}
lda_data_individual['phonation_clean'] = (
    lda_data_individual['phonation']
    .astype(str).str.strip().str.lower()
    .replace(label_alias)
)

valid_labels = ['lenis', 'tense', 'aspirated']
lda_data_individual = lda_data_individual[lda_data_individual['phonation_clean'].isin(valid_labels)].copy()

distance_variances = []
participant_info = []

for participant_id in lda_data_individual['prolific_id'].unique():
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id]

    phonation_counts = participant_data['phonation_clean'].value_counts()
    if len(phonation_counts) < 3 or phonation_counts.min() < 1:
        continue

    centroids = {
        ph: np.array([participant_data[participant_data['phonation_clean'] == ph]['normed_log_vot'].mean(),
                      participant_data[participant_data['phonation_clean'] == ph]['normed_semitone'].mean()])
        for ph in valid_labels
    }

    centroid_coords = np.array([centroids[label] for label in valid_labels])
    distances = [np.linalg.norm(centroid_coords[i] - centroid_coords[j])
                 for i in range(len(valid_labels)) for j in range(i+1, len(valid_labels))]

    distance_variance = np.var(distances)
    distance_variances.append(distance_variance)

    participant_info.append({
        'participant_id': participant_id,
        'age':             participant_data['age'].iloc[0],
        'gender':          participant_data['gender'].iloc[0],
        'distance_variance': distance_variance,
        'mean_distance':   np.mean(distances),
        'distances':       distances
    })

results_df = pd.DataFrame(participant_info)

print(f"\n참가자 수: {len(results_df)}")
print(f"Distance Variance 통계:")
print(f"  Mean: {results_df['distance_variance'].mean():.4f}")
print(f"  Std:  {results_df['distance_variance'].std():.4f}")
print(f"  Min:  {results_df['distance_variance'].min():.4f}")
print(f"  Max:  {results_df['distance_variance'].max():.4f}")

fig, ax = plt.subplots(figsize=(10, 6))
ax.hist(results_df['distance_variance'], bins=20, edgecolor='black', alpha=0.7, color='steelblue')
ax.set_xlabel('Distance Variance', fontsize=12)
ax.set_ylabel('Number of Participants', fontsize=12)
ax.set_title('Distribution of Distance Variance Across Participants', fontsize=14, fontweight='bold')
ax.grid(True, alpha=0.3)

mean_dv = results_df['distance_variance'].mean()
ax.axvline(mean_dv, color='red', linestyle='--', linewidth=2, label=f'Mean: {mean_dv:.4f}')
ax.legend()

plt.tight_layout()

output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_filename = output_dir / 'distance_variance_histogram.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ 히스토그램이 저장되었습니다: {output_filename}")
plt.show()

# 최대/최소 참가자 저장 (다음 셀에서 사용)
max_idx = results_df['distance_variance'].idxmax()
min_idx = results_df['distance_variance'].idxmin()
max_participant = results_df.loc[max_idx]
min_participant = results_df.loc[min_idx]

print(f"\n최대 Distance Variance: P{max_participant['participant_id']} (DV={max_participant['distance_variance']:.4f})")
print(f"최소 Distance Variance: P{min_participant['participant_id']} (DV={min_participant['distance_variance']:.4f})")
print(f"\n✓ Distance Variance Analysis 완료!")

In [ ]:
# ================ 최대/최소 Distance Variance 참가자의 LDA 시각화 (Global Z-scored) ================
print("\n" + "=" * 60)
print("최대/최소 Distance Variance 참가자의 LDA 결정 경계 (Global Z-scored)")
print("=" * 60)

colors_dark  = {'lenis': '#1f77b4', 'tense': '#ff7f0e', 'aspirated': '#2ca02c'}
colors_light = {'lenis': '#aec7e8', 'tense': '#ffbb78', 'aspirated': '#98df8a'}
label_to_num = {'lenis': 0, 'tense': 1, 'aspirated': 2}
valid_labels = ['lenis', 'tense', 'aspirated']

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for idx, (ax, participant_row) in enumerate([(axes[0], max_participant), (axes[1], min_participant)]):
    participant_id   = participant_row['participant_id']
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id].copy()

    X = participant_data[['normed_log_vot', 'normed_semitone']].values
    y = participant_data['phonation_clean'].values

    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)

        y_pred   = lda.predict(X)
        accuracy = accuracy_score(y, y_pred)

        vot_center = (participant_data['normed_log_vot'].max() + participant_data['normed_log_vot'].min()) / 2
        f0_center  = (participant_data['normed_semitone'].max() + participant_data['normed_semitone'].min()) / 2
        half_range = max(
            participant_data['normed_log_vot'].max() - participant_data['normed_log_vot'].min(),
            participant_data['normed_semitone'].max() - participant_data['normed_semitone'].min()
        ) / 2 * 1.1

        x_min, x_max = vot_center - half_range, vot_center + half_range
        y_min, y_max = f0_center  - half_range, f0_center  + half_range

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        Z         = lda.predict(np.c_[xx.ravel(), yy.ravel()])
        Z_numeric = np.array([label_to_num[label] for label in Z]).reshape(xx.shape)

        cmap = plt.matplotlib.colors.ListedColormap(
            [colors_light['lenis'], colors_light['tense'], colors_light['aspirated']])
        ax.contourf(xx, yy, Z_numeric, alpha=0.3, cmap=cmap, levels=2)
        ax.contour(xx, yy, Z_numeric, colors='black', linewidths=1.5, linestyles='solid', levels=2)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                ax.scatter(participant_data.loc[mask, 'normed_log_vot'],
                           participant_data.loc[mask, 'normed_semitone'],
                           c=colors_dark[phonation], alpha=0.7, s=50,
                           edgecolors='black', linewidth=0.5, label=phonation)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                cx = participant_data.loc[mask, 'normed_log_vot'].mean()
                cy = participant_data.loc[mask, 'normed_semitone'].mean()
                ax.scatter(cx, cy, c=colors_dark[phonation], marker='X', s=200,
                           edgecolors='black', linewidth=2, zorder=5)

        participant_age    = participant_data['age'].iloc[0]
        participant_gender = participant_data['gender'].iloc[0]
        dv = participant_row['distance_variance']

        ax.set_xlabel('Normalized log(VOT) (z-score)', fontsize=11)
        ax.set_ylabel('Normalized Semitone (z-score)', fontsize=11)
        ax.set_xlim([x_min, x_max])
        ax.set_ylim([y_min, y_max])
        ax.set_aspect('equal')

        title_prefix = 'Maximum DV' if idx == 0 else 'Minimum DV'
        ax.set_title(f'{title_prefix}\nP{participant_id} ({participant_age}, {participant_gender})\nDV: {dv:.4f}, Acc: {accuracy*100:.1f}%',
                     fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, linewidth=0.5)

    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {str(e)[:50]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)

plt.suptitle('Comparison of Extreme Distance Variance Participants (Global Z-scored)', fontsize=14, fontweight='bold')
plt.tight_layout()

output_filename = output_dir / 'lda_extreme_distance_variance_global_zscore.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
pdf_filename = output_dir / 'lda_extreme_distance_variance_global_zscore.pdf'
plt.savefig(pdf_filename, bbox_inches='tight', facecolor='white')
print(f"\n✓ 비교 그래프가 저장되었습니다: {output_filename}")
print(f"✓ PDF로도 저장되었습니다: {pdf_filename}")
plt.show()

print(f"\n✓ 최대/최소 Distance Variance 참가자 시각화 완료 (Global Z-scored)!")

In [ ]:
# ================ Centroid 좌표 출력 (Global Z-scored) ================
print("=" * 60)
print("Centroid Coordinates per Phonation (Global Z-scored)")
print("=" * 60)

for label, participant_row in [('Maximum DV', max_participant), ('Minimum DV', min_participant)]:
    participant_id   = participant_row['participant_id']
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == participant_id].copy()
    age    = participant_data['age'].iloc[0]
    gender = participant_data['gender'].iloc[0]
    dv     = participant_row['distance_variance']

    print(f"\n{'─'*50}")
    print(f"{label} | P{participant_id} ({age}, {gender}) | DV: {dv:.4f}")
    print(f"{'─'*50}")
    print(f"{'Phonation':<12} {'Norm log(VOT) (z)':>18} {'Norm Semitone (z)':>18}")
    print(f"{'─'*50}")

    for phonation in valid_labels:
        mask = participant_data['phonation_clean'] == phonation
        if mask.sum() == 0:
            continue
        norm_vot = participant_data.loc[mask, 'normed_log_vot'].mean()
        norm_f0  = participant_data.loc[mask, 'normed_semitone'].mean()
        print(f"{phonation:<12} {norm_vot:>18.4f} {norm_f0:>18.4f}")

print(f"\n{'─'*50}")
print("X marker on each plot indicates the centroid position.")

In [ ]:
# ================ VOT-dominant (P216) vs F0-dominant (P625) 비교 ================
print("=" * 60)
print("VOT-dominant (P216) vs F0-dominant (P625) Centroid 비교")
print("=" * 60)

focus_participants = {216: 'VOT-dominant', 625: 'F0-dominant'}

colors_dark  = {'lenis': '#1f77b4', 'tense': '#ff7f0e', 'aspirated': '#2ca02c'}
colors_light = {'lenis': '#aec7e8', 'tense': '#ffbb78', 'aspirated': '#98df8a'}
label_to_num = {'lenis': 0, 'tense': 1, 'aspirated': 2}
valid_labels = ['lenis', 'tense', 'aspirated']

# ── 1. Centroid 좌표 출력 ──────────────────────────────────────────
for pid, role in focus_participants.items():
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == pid].copy()
    if participant_data.empty:
        print(f"\nP{pid} ({role}): 데이터 없음")
        continue
    age    = participant_data['age'].iloc[0]
    gender = participant_data['gender'].iloc[0]
    print(f"\n{'─'*50}")
    print(f"{role} | P{pid} ({age}, {gender})")
    print(f"{'─'*50}")
    print(f"{'Phonation':<12} {'Norm log(VOT) (z)':>18} {'Norm Semitone (z)':>18}")
    print(f"{'─'*50}")
    for phonation in valid_labels:
        mask = participant_data['phonation_clean'] == phonation
        if mask.sum() == 0:
            continue
        norm_vot = participant_data.loc[mask, 'normed_log_vot'].mean()
        norm_f0  = participant_data.loc[mask, 'normed_semitone'].mean()
        print(f"{phonation:<12} {norm_vot:>18.4f} {norm_f0:>18.4f}")

# ── 2. LDA 결정 경계 시각화 (Global Z-scored) ──────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for ax, (pid, role) in zip(axes, focus_participants.items()):
    participant_data = lda_data_individual[lda_data_individual['prolific_id'] == pid].copy()
    if participant_data.empty:
        ax.text(0.5, 0.5, f'P{pid}: 데이터 없음', ha='center', va='center', transform=ax.transAxes)
        continue

    X = participant_data[['normed_log_vot', 'normed_semitone']].values
    y = participant_data['phonation_clean'].values

    try:
        lda = LinearDiscriminantAnalysis()
        lda.fit(X, y)
        y_pred   = lda.predict(X)
        accuracy = accuracy_score(y, y_pred)

        vot_center = (participant_data['normed_log_vot'].max() + participant_data['normed_log_vot'].min()) / 2
        f0_center  = (participant_data['normed_semitone'].max() + participant_data['normed_semitone'].min()) / 2
        half_range = max(
            participant_data['normed_log_vot'].max() - participant_data['normed_log_vot'].min(),
            participant_data['normed_semitone'].max() - participant_data['normed_semitone'].min()
        ) / 2 * 1.1

        x_min, x_max = vot_center - half_range, vot_center + half_range
        y_min, y_max = f0_center  - half_range, f0_center  + half_range

        xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200),
                             np.linspace(y_min, y_max, 200))
        Z         = lda.predict(np.c_[xx.ravel(), yy.ravel()])
        Z_numeric = np.array([label_to_num[lb] for lb in Z]).reshape(xx.shape)

        cmap = plt.matplotlib.colors.ListedColormap(
            [colors_light['lenis'], colors_light['tense'], colors_light['aspirated']])
        ax.contourf(xx, yy, Z_numeric, alpha=0.3, cmap=cmap, levels=2)
        ax.contour(xx, yy, Z_numeric, colors='black', linewidths=1.5, linestyles='solid', levels=2)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                ax.scatter(participant_data.loc[mask, 'normed_log_vot'],
                           participant_data.loc[mask, 'normed_semitone'],
                           c=colors_dark[phonation], alpha=0.7, s=50,
                           edgecolors='black', linewidth=0.5, label=phonation)

        for phonation in valid_labels:
            mask = (y == phonation)
            if mask.sum() > 0:
                cx = participant_data.loc[mask, 'normed_log_vot'].mean()
                cy = participant_data.loc[mask, 'normed_semitone'].mean()
                ax.scatter(cx, cy, c=colors_dark[phonation], marker='X', s=200,
                           edgecolors='black', linewidth=2, zorder=5)

        age    = participant_data['age'].iloc[0]
        gender = participant_data['gender'].iloc[0]
        ax.set_xlabel('Normalized log(VOT) (z-score)', fontsize=11)
        ax.set_ylabel('Normalized Semitone (z-score)', fontsize=11)
        ax.set_xlim([x_min, x_max])
        ax.set_ylim([y_min, y_max])
        ax.set_aspect('equal')
        ax.set_title(f'{role}\nP{pid} ({age}, {gender})\nAcc: {accuracy*100:.1f}%',
                     fontsize=11, fontweight='bold')
        ax.legend(fontsize=9)
        ax.grid(True, alpha=0.3, linewidth=0.5)

    except Exception as e:
        ax.text(0.5, 0.5, f'Error: {str(e)[:50]}',
                ha='center', va='center', transform=ax.transAxes, fontsize=10)

plt.suptitle('VOT-dominant (P216) vs F0-dominant (P625): LDA Decision Boundaries (Global Z-scored)',
             fontsize=13, fontweight='bold')
plt.tight_layout()

output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_filename = output_dir / 'lda_vot_vs_f0_dominant_global_zscore.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
pdf_filename = output_dir / 'lda_vot_vs_f0_dominant_global_zscore.pdf'
plt.savefig(pdf_filename, bbox_inches='tight', facecolor='white')
print(f"\n✓ 저장됨: {output_filename}")
print(f"✓ PDF 저장됨: {pdf_filename}")
plt.show()

In [ ]:
# ================ Optimal f0 weight (w*) per participant ================
# d²(w) = w*(Δsemitone)² + (1-w)*(Δlog_vot)²  →  Var(d²) 최소화
# w* = -Cov(a, b) / Var(a),  a_k = (Δsemitone_k)² - (Δlog_vot_k)²,  b_k = (Δlog_vot_k)²
# [0,1]에 clamp

from itertools import combinations

pairs = list(combinations(valid_labels, 2))  # 3쌍

def compute_w_star(participant_data, vot_col, f0_col):
    centroids = {
        ph: participant_data.loc[participant_data['phonation_clean'] == ph, [vot_col, f0_col]].mean()
        for ph in valid_labels
    }
    a, b = [], []
    for ph1, ph2 in pairs:
        d_vot = centroids[ph1][vot_col] - centroids[ph2][vot_col]
        d_f0  = centroids[ph1][f0_col]  - centroids[ph2][f0_col]
        a.append(d_f0**2 - d_vot**2)
        b.append(d_vot**2)
    a, b = np.array(a), np.array(b)
    var_a = np.var(a, ddof=0)
    if var_a == 0:
        return np.nan
    w = -np.cov(a, b, ddof=0)[0, 1] / var_a
    return float(np.clip(w, 0, 1))

records = []
for pid in lda_data_individual['prolific_id'].unique():
    pdata  = lda_data_individual[lda_data_individual['prolific_id'] == pid]
    counts = pdata['phonation_clean'].value_counts()
    if len(counts) < 3 or counts.min() < 1:
        continue
    records.append({
        'prolific_id': pid,
        'age':         pdata['age'].iloc[0],
        'gender':      pdata['gender'].iloc[0],
        'w_star':      compute_w_star(pdata, 'normed_log_vot', 'normed_semitone'),
    })

w_star_df = pd.DataFrame(records)
print(w_star_df.to_string(index=False))
print(f"\n참가자 수: {len(w_star_df)}")
print(f"\nw* (global z-scored)  mean={w_star_df['w_star'].mean():.3f}, std={w_star_df['w_star'].std():.3f}")

In [ ]:
# ================ P216: Alternative w* — d(lenis, tense) = d(lenis, aspirated) ================
# 기존 방식(분산 최소화)에서 P216은 w*=1.0으로 clip됨 → 대안 목적함수 적용
#
# 목적: w*(Δf0_LT)² + (1-w)*(Δvot_LT)² = w*(Δf0_LA)² + (1-w)*(Δvot_LA)²
#
# d²(p,q; w) = w*α + (1-w)*β  (α=(Δf0)², β=(Δvot)²)
# d²(L,T;w) = d²(L,A;w) 를 w에 대해 풀면 선형 방정식 → closed-form 존재:
#
#   w* = (β_LA - β_LT) / [(α_LT - α_LA) + (β_LA - β_LT)]
#
# [0,1] 내 해가 없으면 grid search로 최소 |d(L,T) - d(L,A)| 지점을 찾음

import numpy as np
import matplotlib.pyplot as plt

pid = 216
p216 = lda_data_individual[lda_data_individual['prolific_id'] == pid].copy()

# ── centroid 계산 ──────────────────────────────────────────────────────
c = {ph: np.array([p216.loc[p216['phonation_clean'] == ph, 'normed_log_vot'].mean(),
                   p216.loc[p216['phonation_clean'] == ph, 'normed_semitone'].mean()])
     for ph in ['lenis', 'tense', 'aspirated']}

print(f"P{pid} Centroids (normed_log_vot, normed_semitone):")
for ph, coord in c.items():
    print(f"  {ph:<12}: vot={coord[0]:.4f},  f0={coord[1]:.4f}")

# ── squared component differences ────────────────────────────────────
alpha_LT = (c['lenis'][1] - c['tense'][1])**2      # (Δf0_LT)²
beta_LT  = (c['lenis'][0] - c['tense'][0])**2      # (Δvot_LT)²
alpha_LA = (c['lenis'][1] - c['aspirated'][1])**2  # (Δf0_LA)²
beta_LA  = (c['lenis'][0] - c['aspirated'][0])**2  # (Δvot_LA)²

print(f"\n  Lenis–Tense    :  Δvot²={beta_LT:.6f},  Δf0²={alpha_LT:.6f}")
print(f"  Lenis–Aspirated:  Δvot²={beta_LA:.6f},  Δf0²={alpha_LA:.6f}")

# ── closed-form solution ──────────────────────────────────────────────
denom = (alpha_LT - alpha_LA) + (beta_LA - beta_LT)
numer = beta_LA - beta_LT

if abs(denom) < 1e-12:
    w_cf = np.nan
    print("\n[Closed-form] 분모=0 → 모든 w에서 d(L,T)=d(L,A) (또는 해 없음)")
else:
    w_cf = numer / denom
    w_cf_clipped = float(np.clip(w_cf, 0.0, 1.0))
    print(f"\n[Closed-form] w* (unconstrained) = {w_cf:.6f}")
    if 0.0 <= w_cf <= 1.0:
        print(f"[Closed-form] → [0,1] 내 해 존재: w* = {w_cf:.6f}")
    else:
        print(f"[Closed-form] → [0,1] 밖 → clipped w* = {w_cf_clipped:.6f}")

# ── grid search (step=0.001) 검증 ────────────────────────────────────
def dist_w(c1, c2, w):
    return np.sqrt(w * (c1[1] - c2[1])**2 + (1 - w) * (c1[0] - c2[0])**2)

ws = np.linspace(0.0, 1.0, 1001)
signed_diff = np.array([dist_w(c['lenis'], c['tense'], w) - dist_w(c['lenis'], c['aspirated'], w)
                        for w in ws])
abs_diff = np.abs(signed_diff)
w_grid   = ws[np.argmin(abs_diff)]

print(f"\n[Grid search step=0.001] best w = {w_grid:.3f},  |d(L,T) - d(L,A)| = {abs_diff.min():.8f}")

# 결론: 어느 방법 사용할지 결정
w_star_alt = w_cf_clipped if not np.isnan(w_cf) else w_grid
print(f"\n★ P{pid} alternative w* = {w_star_alt:.4f}")
print(f"  (기존 variance 방식: 1.0000 — clip)")

# ── 시각화 ──────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 목적함수 |d(L,T;w) - d(L,A;w)|
ax = axes[0]
ax.plot(ws, abs_diff, color='steelblue', linewidth=2, label='|d(L,T;w) − d(L,A;w)|')
ax.axhline(0, color='gray', linestyle=':', linewidth=1)
ax.axvline(w_star_alt, color='red', linestyle='--', linewidth=1.5,
           label=f'w* = {w_star_alt:.4f}{"  (clipped)" if not (0 < w_cf < 1) else ""}')
ax.set_xlabel('w  (f0 weight)', fontsize=12)
ax.set_ylabel('|d(Lenis, Tense) − d(Lenis, Aspirated)|', fontsize=11)
ax.set_title(f'P{pid}: Objective — Equal L–T & L–A Distances', fontsize=11, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)

# 오른쪽: 세 거리 곡선
ax2 = axes[1]
dLT = np.array([dist_w(c['lenis'], c['tense'],      w) for w in ws])
dLA = np.array([dist_w(c['lenis'], c['aspirated'],   w) for w in ws])
dTA = np.array([dist_w(c['tense'], c['aspirated'],   w) for w in ws])
ax2.plot(ws, dLT, color='#1f77b4', linewidth=2, label='d(Lenis, Tense)')
ax2.plot(ws, dLA, color='#2ca02c', linewidth=2, label='d(Lenis, Aspirated)')
ax2.plot(ws, dTA, color='#ff7f0e', linewidth=2, linestyle='--', label='d(Tense, Aspirated)')
ax2.axvline(w_star_alt, color='red', linestyle='--', linewidth=1.5, label=f'w* = {w_star_alt:.4f}')
ax2.set_xlabel('w  (f0 weight)', fontsize=12)
ax2.set_ylabel('Weighted distance', fontsize=12)
ax2.set_title(f'P{pid}: Centroid Distances as a Function of w', fontsize=11, fontweight='bold')
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)

plt.suptitle(f'P{pid}: w* via Equal Lenis–Tense & Lenis–Aspirated Distances\n'
             f'(Closed-form: {w_cf:.4f} → clipped to {w_star_alt:.4f})',
             fontsize=12, fontweight='bold')
plt.tight_layout()

output_dir = Path.cwd() / "output"
output_dir.mkdir(exist_ok=True)
output_filename = output_dir / f'p{pid}_w_star_equal_LT_LA_distances.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ 저장됨: {output_filename}")
plt.show()

In [ ]:
# ================ P216 & P625: w*-transformed coordinate scatter ================
# 좌표 변환: x' = sqrt(1-w) * normed_log_vot,  y' = sqrt(w) * normed_semitone
# → 변환 공간에서의 유클리디안 거리 = 원래 공간의 weighted distance d(·; w)
#
# P216: alternative w* (d(L,T) = d(L,A) 방식)  ← w_star_alt
# P625: original w* (variance 최소화 방식)       ← w_star_df

w216 = w_star_alt
w625 = float(w_star_df.loc[w_star_df['prolific_id'] == 625, 'w_star'].values[0])

print(f"P216  w* (equal-distance method) = {w216:.4f}")
print(f"P625  w* (variance method)        = {w625:.4f}")

colors_dark  = {'lenis': '#1f77b4', 'tense': '#ff7f0e', 'aspirated': '#2ca02c'}
colors_light = {'lenis': '#aec7e8', 'tense': '#ffbb78', 'aspirated': '#98df8a'}
label_to_num = {'lenis': 0, 'tense': 1, 'aspirated': 2}
valid_labels  = ['lenis', 'tense', 'aspirated']

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

for col_idx, (pid, w_val) in enumerate([(216, w216), (625, w625)]):
    pdata = lda_data_individual[lda_data_individual['prolific_id'] == pid].copy()
    y     = pdata['phonation_clean'].values
    age   = pdata['age'].iloc[0]
    gender = pdata['gender'].iloc[0]

    def plot_panel(ax, xs, ys, title_suffix, xlabel, ylabel):
        X = np.column_stack([xs, ys])
        try:
            lda = LinearDiscriminantAnalysis()
            lda.fit(X, y)
            acc = accuracy_score(y, lda.predict(X))

            cx_c = (xs.max() + xs.min()) / 2
            cy_c = (ys.max() + ys.min()) / 2
            hr   = max(xs.max() - xs.min(), ys.max() - ys.min()) / 2 * 1.2
            xlim = (cx_c - hr, cx_c + hr)
            ylim = (cy_c - hr, cy_c + hr)

            xx, yy = np.meshgrid(np.linspace(*xlim, 200), np.linspace(*ylim, 200))
            Z_num  = np.array([label_to_num[lb]
                               for lb in lda.predict(np.c_[xx.ravel(), yy.ravel()])]).reshape(xx.shape)
            cmap   = plt.matplotlib.colors.ListedColormap([colors_light[lb] for lb in valid_labels])
            ax.contourf(xx, yy, Z_num, alpha=0.3, cmap=cmap, levels=2)
            ax.contour( xx, yy, Z_num, colors='black', linewidths=1.2, levels=2)

            for ph in valid_labels:
                mask = (y == ph)
                if mask.sum() == 0:
                    continue
                ax.scatter(xs[mask], ys[mask], c=colors_dark[ph], alpha=0.7, s=40,
                           edgecolors='black', linewidth=0.4, label=ph)
                ax.scatter(xs[mask].mean(), ys[mask].mean(),
                           c=colors_dark[ph], marker='X', s=200,
                           edgecolors='black', linewidth=2, zorder=5)

            ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect('equal')
            ax.set_xlabel(xlabel, fontsize=10)
            ax.set_ylabel(ylabel, fontsize=10)
            ax.set_title(f'P{pid} ({age}, {gender})  {title_suffix}\nAcc: {acc*100:.1f}%',
                         fontsize=10, fontweight='bold')
            ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
        except Exception as e:
            ax.text(0.5, 0.5, str(e), ha='center', va='center', transform=ax.transAxes)

    # 위 행: 원래 공간
    xs_orig = pdata['normed_log_vot'].values
    ys_orig = pdata['normed_semitone'].values
    plot_panel(axes[0, col_idx], xs_orig, ys_orig,
               '[Original space]',
               'Norm. log(VOT) (z)', 'Norm. Semitone (z)')

    # 아래 행: w*-변환 공간
    xs_tr = np.sqrt(1 - w_val) * pdata['normed_log_vot'].values
    ys_tr = np.sqrt(w_val)     * pdata['normed_semitone'].values
    w_label = 'equal-dist.' if pid == 216 else 'var-min.'
    plot_panel(axes[1, col_idx], xs_tr, ys_tr,
               f'[w*={w_val:.4f} ({w_label}) transformed]',
               f'√(1−{w_val:.3f}) · Norm. log(VOT)',
               f'√{w_val:.3f} · Norm. Semitone')

plt.suptitle('P216 & P625: Original vs w*-Transformed Space\n'
             'Euclidean distance in transformed space = weighted distance d(·; w*)',
             fontsize=13, fontweight='bold')
plt.tight_layout()

output_dir = Path.cwd() / "output"
output_filename = output_dir / 'p216_p625_w_star_transformed_scatter.png'
plt.savefig(output_filename, dpi=300, bbox_inches='tight', facecolor='white')
print(f"\n✓ 저장됨: {output_filename}")
plt.show()